## <font color = #f56055> R script

In [ ]:
pacman::p_load(tidyverse, reshape2, ggplot2)

In [ ]:
my_theme = 
    theme(plot.title = element_text(color = "black", size = 20, hjust = 0.5),
        axis.title.x = element_text(color = "black", size = 15),
        axis.title.y = element_text(color = "black", size = 15),
        axis.text.x = element_text(vjust= 0.6,size= 10),
        axis.text.y = element_text(vjust = 1, size = 10),
        legend.key.size = unit(40, "pt"),
        legend.text = element_text(color = "black", size = 15),
        legend.background = element_rect(fill = "white", color = "white"),
        legend.key = element_rect(fill = "white"))

In [ ]:
yas_df_wider = yas_pathway_rpkm_sum %>%
    melt(id.vars = "pathway", variable.name = "sample", value.name = "RPKM") %>%
    left_join(yas_pathway_stragety, by = "pathway") %>%
    group_by(sample, Strategy) %>%
    summarise(mean = sum(RPKM)) %>% 
    pivot_wider(names_from = Strategy, values_from = mean) %>%
    column_to_rownames(var = "sample") %>%
    apply(1, function(x) x/sum(x)) %>% t %>%
    as.data.frame %>%
    rownames_to_column(var = "sample") %>%
    left_join(meta, by = "sample")

yas_df_long = yas_df_wider[.1:4] %>% melt(id.vars = "sample", variable.name = "Strategy", value.name = "value") %>%
    left_join(meta, by = "sample")

#### <font color=lightblue> Fig.3

In [ ]:
yas_df_long %>% 
    ggplot() +
    geom_beeswarm(aes(as.factor(Treatment), value, color = Treatment), alpha = 0.5, size = 3, shape = 16) +
    geom_violin(aes(as.factor(Treatment), value, color = Treatment), alpha = 0.2, size = 1, shape = 16) +
    stat_boxplot(aes(as.factor(Treatment), value), geom = "errorbar",  color = "black", width = .001, alpha = 0.6) +
    stat_summary(aes(as.factor(Treatment), value, fill = Treatment), fun = "mean", geom = "point", size = 4,  pch = 21, color = "black", linewidth = 2) +
    geom_smooth(aes(as.numeric(Treatment), value), method = "lm", color = "#b74539", se = FALSE, linewidth = 2, alpha = .5) +
    facet_wrap(~Strategy, scales = "free") +
    scale_color_manual(values = c("#46bce2", "#18c584", "#ebd07d")) +
    scale_fill_manual(values = c("#46bce2", "#18c584", "#ebd07d")) +
    theme_classic() +
    scale_y_continuous(expand = expansion(mult = c(0.05, 0.3))) +
    my_theme +
    xlab("") +
    ylab("") +
    theme(legend.position = "none", strip.text = element_text(size = 20), strip.background = element_rect(fill = "grey90")) 

In [ ]:
HSD.test(aov(value~Treatment, data = subset(yas_df_long, Strategy == "Growth potential")), "Treatment", group = TRUE)$groups
HSD.test(aov(value~Treatment, data = subset(yas_df_long, Strategy == "Resource acquisition")), "Treatment", group = TRUE)$groups
HSD.test(aov(value~Treatment, data = subset(yas_df_long, Strategy == "Stress tolerance")), "Treatment", group = TRUE)$groups

In [ ]:
glm(value~poly(Treatment, 2), family = gaussian, data = subset(yas_df_long, Strategy == "Growth potential"))
glm(value~poly(Treatment, 1), family = gaussian, data = subset(yas_df_long, Strategy == "Resource acquisition"))
glm(value~poly(Treatment, 2), family = gaussian, data = subset(yas_df_long, Strategy == "Stress tolerance"))

#### <font color=lightblue> Fig.S9

In [ ]:
pacman::p_load(sf, terra, tidyterra)

In [ ]:
china = sf::read_sf("../地图/中国省级地图GS（2019）1719号.geojson") |>
  st_make_valid()
nine_line = sf::read_sf("../地图/九段线GS（2019）1719号.geojson") |>
  st_make_valid()
cli = terra::rast("Beck_KG_V1/Beck_KG_V1_present_0p0083.tif")
names(cli) = "Koppen_Classification"

china_terra = terra::vect(st_transform(china, terra::crs(cli)))
nine_line = st_transform(nine_line, st_crs(china))

cli_china = terra::crop(cli, china_terra)
cli_china = terra::mask(cli_china, china_terra)

cli_smooth = cli_china
for (w in c(3, 5, 7)) {
  cli_smooth = terra::focal(
    cli_smooth,
    w = matrix(1, w, w),
    fun = terra::modal,
    na.rm = TRUE,
    na.policy = "all"
  )
  
  cli_smooth = terra::mask(cli_smooth, china_terra)
}

crs_china = "+proj=aea +lat_1=25 +lat_2=47 +lat_0=0 +lon_0=105 +datum=WGS84 +units=m +no_defs"

cli_plot = terra::project(
  cli_smooth,
  crs_china,
  method = "near"
)

china_plot = st_transform(china, crs_china)
nine_plot = st_transform(nine_line, crs_china)

df = terra::as.data.frame(cli_plot, xy = TRUE, na.rm = TRUE)

colnames(df)[3] = "Koppen_Classification"


df$Koppen_code = as.character(df$Koppen_Classification)

df$Koppen = ifelse(
  df$Koppen_code %in% names(koppen_labels),
  koppen_labels[df$Koppen_code],
  df$Koppen_code
)

df$Koppen = factor(
  df$Koppen,
  levels = c(
    "Af", "Am", "Aw",
    "BWh", "BWk", "BSh", "BSk",
    "Csa", "Csb", "Csc", "Cwa", "Cwb", "Cwc", "Cfa", "Cfb", "Cfc",
    "Dsa", "Dsb", "Dsc", "Dsd", "Dwa", "Dwb", "Dwc", "Dwd",
    "Dfa", "Dfb", "Dfc", "Dfd",
    "ET", "EF"))

koppen_cols = c(
  "Af"  = "#006837",
  "Am"  = "#31A354",
  "Aw"  = "#78C679",
  "BWh" = "#E34A33",
  "BWk" = "#FC8D59",
  "BSh" = "#FDBB84",
  "BSk" = "#FDD49E",
  "Csa" = "#238B45",
  "Csb" = "#41AB5D",
  "Csc" = "#74C476",
  "Cwa" = "#006D2C",
  "Cwb" = "#31A354",
  "Cwc" = "#A1D99B",
  "Cfa" = "#2CA25F",
  "Cfb" = "#66C2A4",
  "Cfc" = "#B2E2E2",
  "Dsa" = "#756BB1",
  "Dsb" = "#9E9AC8",
  "Dsc" = "#BCBDDC",
  "Dsd" = "#DADAEB",
  "Dwa" = "#2171B5",
  "Dwb" = "#6BAED6",
  "Dwc" = "#9ECAE1",
  "Dwd" = "#C6DBEF",
  "Dfa" = "#08519C",
  "Dfb" = "#3182BD",
  "Dfc" = "#6BAED6",
  "Dfd" = "#BDD7E7",
  "ET"  = "#BDBDBD",
  "EF"  = "#F0F0F0")


meta = read.csv("microcosm_meta.csv")
meta_sf = st_as_sf(meta,coords = c("Longitude", "Latitude"), crs = 4326)
meta_sf_proj = st_transform(meta_sf, crs_china)
meta_xy = cbind(meta, st_coordinates(meta_sf_proj)) %>% as.data.frame()

ggplot() +
  geom_raster(data = df,aes(x = x, y = y, fill = Koppen)) +
  geom_sf(data = china_plot,fill = NA,colour = "grey35",linewidth = 0.25) +
  geom_sf(data = st_union(china_plot),fill = NA,colour = "black",linewidth = 0.55) +
  geom_sf(data = nine_plot,colour = "black",linewidth = 0.55,linetype = "longdash") +
  geom_point(data = meta_xy, aes(X,Y),size = 4, fill = "red", colour = "white", stroke = 0.2, alpha = 0.8, pch = 21) +
  scale_fill_manual(values = koppen_cols,drop = FALSE,na.value = "transparent",name = "Köppen climate") +
  coord_sf(crs = crs_china,datum = st_crs(4326),expand = FALSE) +
  theme_minimal(base_size = 12) +
  theme(panel.background = element_rect(fill = "white", colour = NA),
        panel.grid.major = element_line(colour = "grey88", linewidth = 0.35),
        panel.grid.minor = element_blank(),
        axis.title = element_blank(),
        axis.text = element_blank(),
        axis.ticks = element_blank(),
        legend.position = "right",
        legend.title = element_text(size = 10),
        legend.text = element_text(size = 8),
        plot.margin = margin(5, 5, 5, 5))

#### <font color=lightblue> Fig.S10

In [ ]:
yas_df_long %>% subset(SoilType == "AS") %>%
    ggplot() +
    geom_beeswarm(aes(as.factor(Treatment), value, color = Treatment), alpha = 0.5, size = 3, shape = 16) +
    geom_violin(aes(as.factor(Treatment), value, color = Treatment), alpha = 0.2, size = 1, shape = 16) +
    stat_boxplot(aes(as.factor(Treatment), value), geom = "errorbar",  color = "black", width = .001, alpha = 0.6) +
    stat_summary(aes(as.factor(Treatment), value, fill = Treatment), fun = "mean", geom = "point", size = 4,  pch = 21, color = "black", linewidth = 2) +
    geom_smooth(aes(as.numeric(Treatment), value), method = "lm", color = "#b74539", se = FALSE, linewidth = 2, alpha = .5) +
    facet_wrap(~Strategy, scales = "free") +
    scale_color_manual(values = c("#46bce2", "#18c584", "#ebd07d")) +
    scale_fill_manual(values = c("#46bce2", "#18c584", "#ebd07d")) +
    theme_classic() +
    scale_y_continuous(expand = expansion(mult = c(0.05, 0.3))) +
    my_theme +
    xlab("") +
    ylab("") +
    theme(legend.position = "none", strip.text = element_text(size = 20), strip.background = element_rect(fill = "grey90")) 

In [ ]:
yas_df_long %>% subset(SoilType == "NS") %>%
    ggplot() +
    geom_beeswarm(aes(as.factor(Treatment), value, color = Treatment), alpha = 0.5, size = 3, shape = 16) +
    geom_violin(aes(as.factor(Treatment), value, color = Treatment), alpha = 0.2, size = 1, shape = 16) +
    stat_boxplot(aes(as.factor(Treatment), value), geom = "errorbar",  color = "black", width = .001, alpha = 0.6) +
    stat_summary(aes(as.factor(Treatment), value, fill = Treatment), fun = "mean", geom = "point", size = 4,  pch = 21, color = "black", linewidth = 2) +
    geom_smooth(aes(as.numeric(Treatment), value), method = "lm", color = "#b74539", se = FALSE, linewidth = 2, alpha = .5) +
    facet_wrap(~Strategy, scales = "free") +
    scale_color_manual(values = c("#46bce2", "#18c584", "#ebd07d")) +
    scale_fill_manual(values = c("#46bce2", "#18c584", "#ebd07d")) +
    theme_classic() +
    scale_y_continuous(expand = expansion(mult = c(0.05, 0.3))) +
    my_theme +
    xlab("") +
    ylab("") +
    theme(legend.position = "none", strip.text = element_text(size = 20), strip.background = element_rect(fill = "grey90")) 

In [ ]:
HSD.test(aov(value~Treatment, data = subset(yas_df_long, SoilType == "AS" & Strategy == "Growth potential")), "Treatment", group = TRUE)$groups
HSD.test(aov(value~Treatment, data = subset(yas_df_long, SoilType == "AS" & Strategy == "Resource acquisition")), "Treatment", group = TRUE)$groups
HSD.test(aov(value~Treatment, data = subset(yas_df_long, SoilType == "AS" & Strategy == "Stress tolerance")), "Treatment", group = TRUE)$groups
HSD.test(aov(value~Treatment, data = subset(yas_df_long, SoilType == "NS" & Strategy == "Growth potential")), "Treatment", group = TRUE)$groups
HSD.test(aov(value~Treatment, data = subset(yas_df_long, SoilType == "NS" & Strategy == "Resource acquisition")), "Treatment", group = TRUE)$groups
HSD.test(aov(value~Treatment, data = subset(yas_df_long, SoilType == "NS" & Strategy == "Stress tolerance")), "Treatment", group = TRUE)$groups

In [ ]:
glm(value~poly(Treatment, 2), family = gaussian, data = subset(yas_df_long, SoilType == "AS" & Strategy == "Growth potential"))
glm(value~poly(Treatment, 1), family = gaussian, data = subset(yas_df_long, SoilType == "AS" & Strategy == "Resource acquisition"))
glm(value~poly(Treatment, 2), family = gaussian, data = subset(yas_df_long, SoilType == "AS" & Strategy == "Stress tolerance"))
glm(value~poly(Treatment, 2), family = gaussian, data = subset(yas_df_long, SoilType == "NS" & Strategy == "Growth potential"))
glm(value~poly(Treatment, 1), family = gaussian, data = subset(yas_df_long, SoilType == "NS" & Strategy == "Resource acquisition"))
glm(value~poly(Treatment, 2), family = gaussian, data = subset(yas_df_long, SoilType == "NS" & Strategy == "Stress tolerance"))

#### <font color=lightblue> Fig.S11

In [ ]:
yas_df_long %>% select(Treatment,  soiltype, SOC, Rs) %>%
    reshape2::melt(id.vars = c("Treatment", "soiltype")) %>%
    ggplot(aes(Treatment, value)) +
    geom_beeswarm(aes(color = factor(Treatment)), alpha = 0.5, size = 3, shape = 16) +
    geom_violin(aes(group = Treatment, color = factor(Treatment)), alpha = 0.2) +
    stat_boxplot(aes(group = Treatment), geom = "errorbar", color = "black", width = .001, alpha = 0.6) +
    stat_summary(aes(fill = factor(Treatment)), fun = "mean", geom = "point", size = 4, pch = 21, color = "black", linewidth = 2) +
    geom_smooth(method = "lm", color = "#b74539", se = FALSE, linewidth = 2) +
    facet_wrap(~variable, scales = "free") +
    scale_color_manual(values = c("#46bce2", "#18c584", "#ebd07d")) +
    scale_fill_manual(values = c("#46bce2", "#18c584", "#ebd07d")) +
    theme_classic() +
    scale_y_continuous(expand = expansion(mult = c(0.05, 0.3))) +
    my_theme +
    xlab("") +
    ylab("") +
    theme(legend.position = "none", strip.text = element_text(size = 20), strip.background = element_rect(fill = "grey90")) 

In [ ]:
HSD.test(aov(SOC~Treatment, data = yas_df_long), "Treatment", group = TRUE)$groups
HSD.test(aov(Rs~Treatment, data = yas_df_long), "Treatment", group = TRUE)$groups
glm(SOC~poly(Treatment, 1), family = gaussian, data = yas_df_long)
glm(Rs~poly(Treatment, 1), family = gaussian, data = yas_df_long)

#### <font color=lightblue> Table S1

In [ ]:
summary(aov(value ~ input*Tem, data = subset(yas_df_long,  SoilType == "AS" & Strategy == "Growth potential")))
summary(aov(value ~ input*Tem, data = subset(yas_df_long,  SoilType == "AS" & Strategy == "Resource acquisition")))
summary(aov(value ~ input*Tem, data = subset(yas_df_long,  SoilType == "AS" & Strategy == "Stress tolerance")))

#### <font color=lightblue> Table S2

In [ ]:
summary(aov(value ~ input*Tem, data = subset(yas_df_long,  SoilType == "NS" & Strategy == "Growth potential")))
summary(aov(value ~ input*Tem, data = subset(yas_df_long,  SoilType == "NS" & Strategy == "Resource acquisition")))
summary(aov(value ~ input*Tem, data = subset(yas_df_long,  SoilType == "NS" & Strategy == "Stress tolerance")))

#### <font color=lightblue> Table S3

In [ ]:
summary(aov(SOC ~ input*Tem, data = yas_df_long))

#### <font color=lightblue> Table S4

In [ ]:
summary(aov(Rs ~ input*Tem, data = yas_df_long))